# 2. PARSeq fine-tuning

Fine-tune Base/standard Tiny with the permutation objective, or run the staged Fast Tiny recipe. Each stage selects epochs on an internal training holdout and then refits on all training images. See notebook 0 for stage initialization details.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


## All three presets use the same training API

`model: parseq` selects Base; `model: parseq-tiny` selects Tiny. Base and standard Tiny use 64×256 inputs and permutation-objective fine-tuning. The matching official weights download automatically. Never resume a different model or recipe in an existing output folder. Choose `recipe: fast_tiny` through `configs/alpr_fast_tiny.yaml` to enable the staged parallel-adaptation recipe and single-graph runtime. Its default output capacity is 12 characters plus EOS.


In [ ]:
import torch
from parseq_ocr.models import download_pretrained

print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if config["device"] == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA kernel, or set device: cpu and precision: fp32")
pretrained = download_pretrained(config["model"], config["pretrained_dir"])
print("Verified pretrained weights:", pretrained.name)

## Training

Default recipe: 64×256 input, batch 64, AdamW at 1e-4, BF16, 20 selection epochs, clip norm 1.0. Change settings in the YAML. Use a new `output_dir` when changing the recipe. Resume loads your own local optimizer state; worker augmentation streams may differ after a restart.

In [ ]:
from parseq_ocr.training import train
selection = train(config, resume=True)
print(selection)

In [ ]:
import json
import matplotlib.pyplot as plt

history = [json.loads(line) for line in (config["output_dir"] / "selection" / "history.jsonl").read_text().splitlines()]
plt.plot([row["epoch"] for row in history], [100*row["holdout"]["exact_accuracy"] for row in history], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Internal holdout exact accuracy (%)")
plt.grid(alpha=.2)
plt.show()
print("Final checkpoint:", config["output_dir"] / "full_fit" / "model.pt")